# 1. What you'll learn

        - implement reparameterized latent sampling
- balance reconstruction against KL regularization
- traverse a two-dimensional latent space

        **The one question this notebook answers:** How does a VAE turn an autoencoder bottleneck into a smooth distribution you can sample?

# 2. Setup

This lesson keeps variational autoencoders small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Real 8×8 handwritten digits are scaled to [0,1]. A two-dimensional latent space is intentionally restrictive so you can see its organization and sampling failures.

**Small example:** If an encoder returns μ=2 and log variance=0, sampling z=μ+exp(0/2)ε with ε=-0.5 gives z=1.5 while gradients still flow through μ and variance.

In [ ]:
d=load_digits();X=(d.data/16).astype("float32");y=d.target;Xa,Xb,ya,yb=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED);print("Train/test:",Xa.shape,Xb.shape,"classes:",np.bincount(ya).tolist(),"range:",(X.min(),X.max()))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack([Xa[i].reshape(8,8) for i in range(8)]),cmap="gray_r");axes[0].axis("off");axes[0].set_title("Real digits")
axes[1].hist(Xa.mean(1),bins=25,color="#176b66");axes[1].set_title("Image brightness");axes[2].bar(range(10),np.bincount(ya),color="#d38b45");axes[2].set_title("Class balance");plt.tight_layout();plt.show()

**Takeaway:** Digits share strokes but vary enough to require a distributed representation.

**Takeaway:** Brightness is one easy latent factor but not the full identity.

**Takeaway:** Balanced classes stop the latent map being dominated by one digit.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
loader=DataLoader(TensorDataset(torch.tensor(Xa)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));test_x=torch.tensor(Xb);print("Batch:",next(iter(loader))[0].shape)

# 6. Train

        Training turns the assumptions behind variational autoencoders into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Encode each image into latent mean μ and log variance.
2. Sample ε from a standard normal and form z=μ+σε.
3. Decode z into pixel probabilities.
4. Minimize reconstruction error plus KL distance from a standard-normal latent prior.

In [ ]:
torch.manual_seed(SEED)
class VAE(nn.Module):
 def __init__(self):
  super().__init__();self.enc=nn.Sequential(nn.Linear(64,32),nn.ReLU());self.mu=nn.Linear(32,2);self.logv=nn.Linear(32,2);self.dec=nn.Sequential(nn.Linear(2,32),nn.ReLU(),nn.Linear(32,64),nn.Sigmoid())
 def forward(self,x):
  h=self.enc(x);mu=self.mu(h);logv=self.logv(h);z=mu+torch.exp(.5*logv)*torch.randn_like(mu);return self.dec(z),mu,logv
vae=VAE();opt=torch.optim.Adam(vae.parameters(),lr=.004);losses=[];kls=[]
for epoch in range(28):
 total=kl_total=0
 for (xb0,) in loader:
  opt.zero_grad();r,mu,logv=vae(xb0);rec=nn.functional.binary_cross_entropy(r,xb0,reduction="sum")/len(xb0);kl=-.5*torch.mean(torch.sum(1+logv-mu.pow(2)-logv.exp(),1));loss=rec+.25*kl;loss.backward();opt.step();total+=loss.item();kl_total+=kl.item()
 losses.append(total/len(loader));kls.append(kl_total/len(loader))
print("Loss first/last:",round(losses[0],3),round(losses[-1],3),"last KL:",round(kls[-1],3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
vae.eval()
with torch.no_grad(): h=vae.enc(test_x);mu=vae.mu(h);recon=vae.dec(mu);grid=torch.tensor([[a,b] for a in np.linspace(-2.5,2.5,8) for b in np.linspace(-2.5,2.5,8)],dtype=torch.float32);samples=vae.dec(grid)
mean_base=torch.tensor(Xa.mean(0))[None,:].repeat(len(Xb),1);rmse=torch.sqrt(torch.mean((recon-test_x)**2)).item();base=torch.sqrt(torch.mean((mean_base-test_x)**2)).item()
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(losses,label="total");axes[0].plot(kls,label="KL");axes[0].legend();axes[0].set_title("VAE objectives")
axes[1].scatter(mu[:,0],mu[:,1],c=yb,cmap="tab10",s=10);axes[1].set_title("Posterior means")
canvas=np.block([[samples[(r*8+c)].reshape(8,8).numpy() for c in range(8)] for r in range(8)]);axes[2].imshow(canvas,cmap="gray_r");axes[2].axis("off");axes[2].set_title("Latent traversal");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Reconstruction RMSE={rmse:.3f}; mean-image baseline={base:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Excess KL pressure makes every input encode nearly the same prior distribution, so the decoder ignores z. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
collapsed=VAE();collapsed.load_state_dict(vae.state_dict());opt2=torch.optim.Adam(collapsed.parameters(),lr=.003)
for _ in range(8):
 for (xb0,) in loader:
  opt2.zero_grad();r,mu0,logv0=collapsed(xb0);rec=nn.functional.binary_cross_entropy(r,xb0,reduction="sum")/len(xb0);kl=-.5*torch.mean(torch.sum(1+logv0-mu0.pow(2)-logv0.exp(),1));loss=rec+20*kl;loss.backward();opt2.step()
with torch.no_grad(): hh=collapsed.enc(test_x);cmu=collapsed.mu(hh)
print(f"Healthy latent variance={mu.var(0).mean().item():.3f}; excessive-KL latent variance={cmu.var(0).mean().item():.3f} (posterior collapse)")

**Use this when…** you need a sampleable smooth latent representation and accept blur or likelihood assumptions.

        **Don't use this when…** sharp sample fidelity is primary, latent semantics must be guaranteed, or posterior collapse is unchecked.

        ## Try this

        1. Change beta from .01 to 2 and chart KL versus reconstruction.
2. Use 8 latent dimensions and interpolate two digits.
3. Sample far outside ±3 and inspect off-manifold outputs.